# Q&A SYSTEM WITH TRANSFORMERS

A Question Answering (Q&A) system built using Transformer-based models. The project focuses on fine-tuning pre-trained models to extract or generate accurate answers from a given text context.


# FINE TUNING STEP

## Choix du model et du dataset:

### model
https://huggingface.co/distilbert/distilbert-base-cased

### datasets
trivia_qa : https://huggingface.co/datasets/mandarjoshi/trivia_qa
squad : https://huggingface.co/datasets/rajpurkar/squad_v2
natural_question : https://huggingface.co/datasets/google-research-datasets/natural_questions

## install dependencies

In [ ]:
!pip install transformers
!pip install torch
!pip install datasets
!pip install huggingface_hub
!pip install evaluate

## Dataset load

In [ ]:
HF_TOKEN = "your_hf_token"

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad_v2", split="train")

In [ ]:
ds = ds.train_test_split(test_size=0.2)

In [ ]:
ds["train"][0]

{'id': '56dc7e1b14d3a41400c2691a',
 'title': 'Comprehensive_school',
 'context': 'Since the 1988 Education Reform Act, parents have a right to choose which school their child should go to or whether to not send them to school at all and to home educate them instead. The concept of "school choice" introduces the idea of competition between state schools, a fundamental change to the original "neighbourhood comprehensive" model, and is partly intended as a means by which schools that are perceived to be inferior are forced either to improve or, if hardly anyone wants to go there, to close down. Government policy is currently promoting \'specialisation\' whereby parents choose a secondary school appropriate for their child\'s interests and skills. Most initiatives focus on parental choice and information, implementing a pseudo-market incentive to encourage better schools. This logic has underpinned the controversial league tables of school performance.',
 'question': 'In what year was the 

In [ ]:
ds['train']['question']

Column(['In what year was the Education Reform Act made into law?', 'What other cases does the Supreme Court consider besides disputes between individuals and administrative organs?', ' What is one inconvenience of using surface mount components?', 'HTML and XHTML should be what by all browsers?', 'What did an official with the Seismological Bureau deny receiving?', ...])

In [ ]:
ds['train']['context']

Column(['Since the 1988 Education Reform Act, parents have a right to choose which school their child should go to or whether to not send them to school at all and to home educate them instead. The concept of "school choice" introduces the idea of competition between state schools, a fundamental change to the original "neighbourhood comprehensive" model, and is partly intended as a means by which schools that are perceived to be inferior are forced either to improve or, if hardly anyone wants to go there, to close down. Government policy is currently promoting \'specialisation\' whereby parents choose a secondary school appropriate for their child\'s interests and skills. Most initiatives focus on parental choice and information, implementing a pseudo-market incentive to encourage better schools. This logic has underpinned the controversial league tables of school performance.', 'In Sweden, the Supreme Court and the Supreme Administrative Court respectively function as the highest cour

In [ ]:
ds["train"]["answers"]

Column([{'text': ['1988'], 'answer_start': [10]}, {'text': [], 'answer_start': []}, {'text': [], 'answer_start': []}, {'text': ['rendered in the same way'], 'answer_start': [321]}, {'text': ['reports predicting the earthquake'], 'answer_start': [1290]}, ...])

## Prepocessing

In [ ]:
checkpoint = "distilbert/distilbert-base-cased"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        # Handle cases where answer_start is an empty list (unanswerable questions in SQuAD v2)
        if len(answer["answer_start"]) == 0:
            start_positions.append(0) # Conventionally set to 0 for unanswerable
            end_positions.append(0)   # Conventionally set to 0 for unanswerable
        else:
            start_char = answer["answer_start"][0]
            end_char = answer["answer_start"][0] + len(answer["text"][0])
            sequence_ids = inputs.sequence_ids(i)

            # Find the start and end of the context
            idx = 0
            while sequence_ids[idx] != 1:
                idx += 1
            context_start = idx
            while sequence_ids[idx] == 1:
                idx += 1
            context_end = idx - 1

            # If the answer is not fully inside the context, label it (0, 0)
            if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
                start_positions.append(0)
                end_positions.append(0)
            else:
                # Otherwise it's the start and end token positions
                idx = context_start
                while idx <= context_end and offset[idx][0] <= start_char:
                    idx += 1
                start_positions.append(idx - 1)

                idx = context_end
                while idx >= context_start and offset[idx][1] >= end_char:
                    idx -= 1
                end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
tokenized_squad = ds.map(preprocess_function, batched=True, remove_columns=ds['train'].column_names)

Map:   0%|          | 0/104255 [00:00<?, ? examples/s]

Map:   0%|          | 0/26064 [00:00<?, ? examples/s]

In [ ]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

## TRAINING

In [ ]:
!pip install accelerate

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-base-cased-squad-qa",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad['train'],
    eval_dataset=tokenized_squad['test'],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,1.316325,1.212159
2,1.042537,1.132717
3,0.827823,1.146512


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=19548, training_loss=1.195164172127858, metrics={'train_runtime': 2237.557, 'train_samples_per_second': 139.781, 'train_steps_per_second': 8.736, 'total_flos': 3.064778762363136e+16, 'train_loss': 1.195164172127858, 'epoch': 3.0})

In [ ]:
trainer.push_to_hub("distilbert-base-cased-squad-finetuned")

NameError: name 'trainer' is not defined

## Evaluate

## Inference